# Emily Observability — EINHORN_INDUSTRIAL Colab Interface

This notebook connects a Colab session to the EINHORN_INDUSTRIAL stack via IDUNA.

**What it does:**
- Authenticates as a named Emily cluster (`EMILY_PRIME_COLAB_1` or custom)
- Posts observations and completions to the Apple audit trail
- Submits requirements to Emily Prime via HEIMDAL
- Tails the Apple stream in real time (polling)
- Uploads training artifacts to Google Drive
- Shows current HEIMDAL sprint status

**Required Colab Secrets** (click the padlock icon):
```
IDUNA_BASE_URL     http://your-server:8080
IDUNA_EMAIL        webmaster@localhost      (or operator email)
IDUNA_PASSWORD     your-password
```

Or use agent auth:
```
IDUNA_AGENT_NAME   EMILY_PRIME_COLAB_1
IDUNA_AGENT_SECRET your-agent-secret
```

In [ ]:
# Install einhorn_sdk from IDUNA repo
# In production, replace with: pip install einhorn_sdk
!pip install -q requests
import subprocess, sys
# If running from a clone of the IDUNA repo:
# sys.path.insert(0, '/path/to/IDUNA/sdk/python')
# Or install directly:
# !pip install git+https://github.com/emilyspringerton/IDUNA.git#subdirectory=sdk/python

In [ ]:
from einhorn_sdk.colab import ColabEmily

# Auto-detects secrets from Colab Secrets panel or environment variables
emily = ColabEmily.from_colab_secrets(cluster_id="colab-1")
print(f"Connected as cluster: {emily.cluster_name}")
print(f"Authenticated: {emily.client.is_authenticated()}")

## Health Check + System Status

In [ ]:
# IDUNA health
health = emily.client.health.check()
print(f"IDUNA health: {health}")

# Recent Apples
print("\n--- Last 5 Apples ---")
apples = emily.client.apples.list(limit=5)
for a in apples:
    print(f"  #{a['id']} [{a['type']}] {a['title']}")

In [ ]:
# Current HEIMDAL sprint queue
print("--- HEIMDAL Sprints (pending/in_progress) ---")
sprints = emily.client.heimdal.list(status="pending", limit=10)
if not sprints:
    print("  (no pending sprints)")
for s in sprints:
    print(f"  #{s['id']} [{s['status']}] {s['requirement'][:80]}")

## Post Observations + Submit Work to Emily Prime

In [ ]:
# Post an observation from this Colab session
apple = emily.observe("Colab session started. GPU: T4, RAM: 12GB. Ready for training run.")
print(f"Filed Apple #{apple['id']}")

In [ ]:
# Submit a requirement to Emily Prime via HEIMDAL
# Emily Prime will pick this up on the next RSI cycle and implement it
sprint = emily.request("Analyze the latest GPT-2 training loss curve and suggest hyperparameter changes")
print(f"Sprint #{sprint['id']} submitted: {sprint['status']}")
print(f"Emily Prime will pick this up on the next RSI cycle (within 5 minutes).")

## Training Loop with Apple Hooks

In [ ]:
# Example: wrap a training loop with Emily observability hooks

def training_loop_with_emily(emily, num_epochs=3):
    """
    Example training loop that posts Apples at key milestones.
    Replace the print statements with real training code.
    """
    emily.observe(f"Training started: {num_epochs} epochs planned")

    best_loss = float('inf')
    for epoch in range(1, num_epochs + 1):
        # --- your training code here ---
        import random
        loss = 4.5 - epoch * 0.3 + random.uniform(-0.1, 0.1)
        # --------------------------------

        print(f"Epoch {epoch}/{num_epochs}: loss={loss:.4f}")

        if loss < best_loss:
            best_loss = loss
            emily.observe(
                f"New best loss: {loss:.4f} at epoch {epoch}",
                source_repo="gpt2-alpine-c"
            )

    emily.complete(
        f"Training complete: {num_epochs} epochs, best_loss={best_loss:.4f}",
        body=f"Final loss: {loss:.4f}. Model ready for evaluation.",
        source_repo="gpt2-alpine-c"
    )
    return best_loss

# Run it
best = training_loop_with_emily(emily, num_epochs=3)
print(f"Best loss: {best:.4f}")

## Upload Artifacts to Google Drive

In [ ]:
# Upload a checkpoint or artifact to Google Drive via IDUNA
import os, json, datetime

# Create a sample artifact (replace with your real checkpoint)
artifact = {
    "epoch": 3,
    "loss": 4.1,
    "timestamp": datetime.datetime.utcnow().isoformat(),
    "cluster": emily.cluster_name,
}
artifact_bytes = json.dumps(artifact, indent=2).encode()

filename = f"checkpoint-{datetime.datetime.utcnow().strftime('%Y%m%d-%H%M%S')}.json"
result = emily.client.drive.upload(filename, artifact_bytes, mime_type="application/json")
print(f"Uploaded: {result.get('name')} → {result.get('web_view_link') or result.get('id')}")

## Live Apple Stream (polling tail)

In [ ]:
# Tail the Apple stream for 30 seconds
# Shows new Apples as they arrive (polls every 5s)
emily.tail_apples(n=5, poll_interval=5.0, duration_seconds=30.0)

## Distributed Emily — Git Discipline

In [ ]:
from einhorn_sdk.colab import git_pull_rebase

# Always pull-rebase before pushing from Colab
# This is the S44-05 discipline: multiple Emily clusters can push to the same repo
# without stepping on each other by always rebasing first.
#
# If a conflict occurs, the push is skipped and an escalation Apple is filed.

repo_path = "/content/EMILY"  # mount your repo here
if os.path.exists(repo_path):
    ok = git_pull_rebase(repo_path)
    if not ok:
        emily.escalate("Git pull-rebase conflict detected in EMILY repo from Colab session")
else:
    print(f"Repo not mounted at {repo_path} — skip git pull-rebase")

## User Management (webmaster only)

In [ ]:
# List all local users (requires webmaster token)
users = emily.client.users.list()
print(f"Local users ({len(users)}):")
for u in users:
    print(f"  uid={u['local_uid']:3d}  {u['email']:<30} {u['status']} ({u['display_name']})")

In [ ]:
# Create a new operator user
# new_user = emily.client.users.create(
#     email="operator@yourdomain.com",
#     password="secure-password-here",
#     display_name="Ops Operator"
# )
# print(f"Created: uid={new_user['local_uid']} {new_user['email']}")